# 05 - Data Contract e Bad Batch
## Lending Club | Pandera

**Tech Challenge Fase 04 · Etapa 1: Validação de Dados e Contratos**

**Objetivo:** garantir que lotes com formato ou valores inválidos **não entrem no modelo**. O contrato valida as **23 features produzidas pelo `engineer_features` e `split_features_target`**, imediatamente antes do `ColumnTransformer` (não o CSV original com 151 colunas e nem as 44 features pós-encoding).

Evidências: lote válido aprovado, lote inválido bloqueado, relatório de erros e teste de que a função de predição nem é chamada.

> Para execução em produção: extrair as mesmas 23 features do fluxo de entrada, aplicar as transformações determinísticas e só então chamar `validate_model_input()`. O contrato, o pipeline e os metadados de threshold devem acompanhar a mesma versão do modelo.

## 0. Imports e identidade visual

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandera.errors import SchemaErrors

from credit_scoring_observability.data_contract import (
    MODEL_FEATURES, NUMERIC_FEATURES, CATEGORICAL_FEATURES,
    MODEL_INPUT_SCHEMA, validate_model_input,
    predict_proba_validated, predict_classes_validated,
)

COLORS = {"blue":"#377DFF", "teal":"#00A6A6", "coral":"#FF6B6B", "purple":"#7B61FF", "gold":"#F4B942"}
plt.rcParams.update({"figure.figsize":(9,4.5),"axes.grid":True,"grid.alpha":0.18,"axes.spines.top":False,"axes.spines.right":False,"axes.titlesize":13,"axes.titleweight":"bold"})

## 1. Escopo e regras do contrato

As regras combinam tipagem, obrigatoriedade, domínio e consistência. `annual_inc` é obrigatório e não negativo; `term` admite 36 ou 60; `fico_avg` deve estar entre 300 e 850; variáveis contáveis/saldos não podem ser negativos; `emp_length_missing` deve corresponder ao código `-1` de `emp_length_years`. `dti` pode ser ausente após a conversão determinística de valores inválidos para `NaN`, porque o pipeline aprende a mediana usando apenas o treino.

Não limitamos `revol_util` a 100% porque valores acima de 100% podem representar utilização acima do limite. Também não há teste de unicidade sobre as 23 features, porque diferentes empréstimos podem compartilhar as mesmas características sem termos um identificador contratual único.

In [ ]:
contract_overview = pd.DataFrame({
    "Grupo": ["Numéricas", "Categóricas", "Total"],
    "Quantidade": [len(NUMERIC_FEATURES), len(CATEGORICAL_FEATURES), len(MODEL_FEATURES)],
})
display(contract_overview)
fig, ax = plt.subplots(figsize=(6,4))
bars=ax.bar(["Numéricas","Categóricas"],[len(NUMERIC_FEATURES),len(CATEGORICAL_FEATURES)],color=[COLORS["blue"],COLORS["purple"]],width=.6)
for b in bars: ax.text(b.get_x()+b.get_width()/2,b.get_height()+.3,str(int(b.get_height())),ha="center",weight="bold")
ax.set(ylabel="Número de features",title="Cobertura do Data Contract",ylim=(0,len(NUMERIC_FEATURES)+4));plt.tight_layout();plt.show()

## 2. Obter um lote válido (sem ler milhões de linhas)

Se `data/reference/reference_dataset.csv` estiver disponível, utilizamos 5 registros reais da referência. Caso não esteja, o notebook usa **registros sintéticos explícitos** para demonstrar a regra; o teste do contrato não depende da publicação do CSV no GitHub.

In [ ]:
REFERENCE_PATH = Path("../data/reference/reference_dataset.csv")
if REFERENCE_PATH.is_file():
    reference_preview = pd.read_csv(REFERENCE_PATH, nrows=100)
    good_batch = (
        reference_preview.loc[:, MODEL_FEATURES]
        .dropna(subset=["annual_inc"])
        .head(5)
        .copy()
    )
    source = "Reference Dataset local"
else:
    example = {
        "loan_amnt":12000., "term":36, "home_ownership":"RENT", "annual_inc":65000.,
        "verification_status":"Verified", "purpose":"debt_consolidation", "dti":17.6,
        "delinq_2yrs":0, "inq_last_6mths":1, "open_acc":11, "pub_rec":0,
        "revol_bal":11130., "revol_util":52.2, "total_acc":23,
        "collections_12_mths_ex_med":0, "acc_now_delinq":0,
        "pub_rec_bankruptcies":0, "tax_liens":0, "application_type":"Individual",
        "emp_length_years":5, "emp_length_missing":0, "fico_avg":692.,
        "credit_history_years":14.7,
    }
    good_batch=pd.DataFrame([example]*5,columns=MODEL_FEATURES)
    good_batch.loc[1,"loan_amnt"]=15000.
    good_batch.loc[2,"fico_avg"]=710.
    source="Lote sintético (reference_dataset.csv não localizado)"
assert len(good_batch)>0, "Nenhum registro disponível para o lote de teste."
print("Origem:",source,"| Shape:",good_batch.shape)
display(good_batch.head())

## 3. Execução com lote válido

In [ ]:
validated_batch=validate_model_input(good_batch)
assert validated_batch.shape[1] == 23
print("APROVADO: lote válido passou pelo contrato.")
print("Registros:",len(validated_batch),"| Features:",validated_batch.shape[1])

## 4. Criar um Bad Batch intencional

Os erros são introduzidos **depois da feature engineering** para que o contrato os detecte. São exemplos: valor de empréstimo negativo, prazo fora do domínio, renda nula, FICO acima do intervalo e inconsistência entre anos de emprego e indicador de ausência.

In [ ]:
bad_batch=good_batch.copy(deep=True)
# Não dependemos de haver cinco registros distintos, apenas >=1; use a mesma linha
# para vários erros caso o lote tenha tamanho reduzido.
bad_batch.iloc[0,bad_batch.columns.get_loc("loan_amnt")]=-500.
bad_batch.iloc[0,bad_batch.columns.get_loc("term")]=48
bad_batch.iloc[0,bad_batch.columns.get_loc("annual_inc")]=np.nan
bad_batch.iloc[0,bad_batch.columns.get_loc("fico_avg")]=910.
bad_batch.iloc[0,bad_batch.columns.get_loc("emp_length_missing")]=1
bad_batch.iloc[0,bad_batch.columns.get_loc("emp_length_years")]=5
BAD_BATCH_PATH=Path("../data/invalid/bad_batch.csv")
BAD_BATCH_PATH.parent.mkdir(parents=True,exist_ok=True)
bad_batch.to_csv(BAD_BATCH_PATH,index=False)
print("Lote inválido salvo localmente:",BAD_BATCH_PATH)
display(bad_batch[["loan_amnt","term","annual_inc","fico_avg","emp_length_years","emp_length_missing"]].head())

## 5. Demonstrar falha e listar os erros

In [ ]:
try:
    validate_model_input(bad_batch)
except SchemaErrors as error:
    print("BLOQUEADO: contrato rejeitou o Bad Batch antes da inferência.")
    failures=error.failure_cases.copy()
    display(failures.head(30))
else:
    raise AssertionError("ERRO: o lote inválido foi aceito pelo contrato!")

In [ ]:
if "column" in failures.columns:
    fail_counts=(failures["column"].fillna("Regra entre colunas").astype(str).value_counts())
    fig,ax=plt.subplots(figsize=(9,4.5))
    ax.barh(fail_counts.index[::-1],fail_counts.values[::-1],color=COLORS["coral"])
    ax.set(xlabel="Ocorrências detectadas",title="Regras violadas no Bad Batch")
    plt.tight_layout();plt.show()

## 6. Confirmar que a predição não foi executada

O teste abaixo usa um modelo sentinela. Se Pandera bloquear o lote corretamente, o método `predict_proba()` **não será chamado**. Esta é a demonstração central pedida no desafio.

In [ ]:
class PredictionSpy:
    def __init__(self): self.calls=0
    def predict_proba(self, X):
        self.calls+=1
        return np.tile([0.75,0.25],(len(X),1))

spy=PredictionSpy()
try:
    predict_proba_validated(spy,bad_batch)
except SchemaErrors:
    print("Lote bloqueado. Chamadas ao modelo:",spy.calls)
else:
    raise AssertionError("Bad Batch chegou ao modelo")
assert spy.calls==0

## 7. Inferência real opcional com V2

Esta seção usa o arquivo `.joblib` e o threshold versionado no JSON da V2, **se ambos estiverem presentes**. `pipeline.predict()` sozinho aplica 0,50, portanto aqui a classe V2 é construída explicitamente com `predict_proba >= decision_threshold`. Se os artefatos ainda não estiverem disponíveis, as evidências do contrato continuam válidas.

In [ ]:
MODEL_PATH=Path("../models/credit_scoring_optimized.joblib")
META_PATH=Path("../models/credit_scoring_optimized_metadata.json")
if MODEL_PATH.is_file() and META_PATH.is_file():
    pipeline=joblib.load(MODEL_PATH)
    model_meta=json.loads(META_PATH.read_text(encoding="utf-8"))
    threshold=float(model_meta["decision_threshold"])
    probabilities=predict_proba_validated(pipeline,good_batch)
    classes=(probabilities>=threshold).astype(int)
    print("Threshold V2:",threshold)
    display(pd.DataFrame({"probabilidade_default":probabilities,"classe_v2":classes}))
else:
    print("Inferência opcional não executada: modelo/metadata V2 ausentes.")

## 8. Conclusão

- O contrato exige 23 features pós-feature engineering, pré-encoding.
- Lotes válidos são aprovados e podem seguir para a predição.
- Erros de domínio, obrigatoriedade e consistência geram `SchemaErrors` com relatório.
- O Bad Batch é salvo apenas na pasta `data/`, que permanece ignorada pelo Git.
- O modelo sentinela comprova que o lote inválido não entra no `predict_proba`.

**Próxima atividade:** executar `uv run pytest tests/test_data_contract.py -v`, incorporar o validador no caminho oficial de inferência e documentar a interface do schema no README.